In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [3]:
df = pd.read_csv("../data/sintetis/dataset_irigasi.csv")
# df = pd.read_csv("../data/dataset_irigasi.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 5000 baris, 10 kolom


,jam,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec,irrigation_action
0,0,55.0,27.6,25.2,78.9,62.7,86.9,104.9,2.50,0
1,1,52.2,26.1,24.6,80.2,64.5,91.1,102.7,2.33,0
2,2,50.5,25.8,24.3,76.8,62.9,92.3,107.9,2.25,0
3,3,49.6,25.0,24.6,79.5,62.1,92.0,110.8,2.24,0
4,4,49.4,26.5,23.9,78.9,63.5,88.0,114.1,2.23,0


# Pisahkan Fitur & Label

In [4]:
FEATURES = ["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]

X = df[FEATURES].values
y = df["irrigation_action"].values

print(f"Fitur: {FEATURES}")
print(f"Label 0 (tidak siram): {(y==0).sum()}")
print(f"Label 1 (siram):       {(y==1).sum()}")
print(f"Rasio siram: {y.mean()*100:.1f}%")

Fitur: ['soil_moisture', 'soil_temperature', 'air_temperature', 'air_humidity']
Label 0 (tidak siram): 4375
Label 1 (siram):       625
Rasio siram: 12.5%


In [5]:
if len(set(y)) < 2:
    print("⚠️ BAHAYA: cuma 1 kelas! Data belum variatif, model gak bisa dilatih.")
else:
    print(f"OK — {len(set(y))} kelas. Lanjut.")

OK — 2 kelas. Lanjut.


# Split Train/Test

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 4000, Test: 1000


# Training Random Forest

In [8]:
model = RandomForestClassifier(
    n_estimators=15,     
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1: 0.5054 (+/- 0.0175)


# Evaluasi di Test Set

In [10]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Tidak siram', 'Siram'])}")

Accuracy: 0.7880
F1: 0.5000

Confusion Matrix:
[[682 193]
 [ 19 106]]

              precision    recall  f1-score   support

 Tidak siram       0.97      0.78      0.87       875
       Siram       0.35      0.85      0.50       125

    accuracy                           0.79      1000
   macro avg       0.66      0.81      0.68      1000
weighted avg       0.90      0.79      0.82      1000



# Feature importance

In [11]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:18s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  air_temperature    0.3182  ███████████████
  soil_temperature   0.3065  ███████████████
  air_humidity       0.2576  ████████████
  soil_moisture      0.1177  █████


# Test Prediksi Manual

In [12]:
# Skenario 1: tanah normal (62%) tapi panas & kering → harusnya SIRAM
s1 = [[62, 35, 38, 40]]
# Skenario 2: tanah agak kering (58%) tapi lembab & sejuk → harusnya TUNDA
s2 = [[58, 24, 25, 90]]

for i, s in enumerate([s1, s2], 1):
    pred = model.predict(s)[0]
    conf = model.predict_proba(s)[0].max()
    print(f"Skenario {i}: {s[0]} → {'SIRAM' if pred else 'TIDAK'} ({conf:.1%})")

Skenario 1: [62, 35, 38, 40] → SIRAM (56.4%)
Skenario 2: [58, 24, 25, 90] → SIRAM (61.0%)


# Simpan Model

In [13]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/rf_irigasi.joblib")
print("Model tersimpan: models/rf_irigasi.joblib")

Model tersimpan: models/rf_irigasi.joblib


# Convert ke esp

In [ ]:
from micromlgen import port

c_code = port(model)
with open("model_irigasi.h", "w") as f:
    f.write(c_code)
print("Tersimpan: model_irigasi.h (siap #include di Arduino IDE)")
print(f"Ukuran: {len(c_code)} karakter")

Tersimpan: model_irigasi.h (siap #include di Arduino IDE)
Ukuran: 348702 karakter
